# Q1 and Q2, step by step

Two questions

- does an author go back to a journal she already published in, more often than chance
- does she stay with the same publisher, more often than chance

Both answers have the same shape, one number, the observed rate divided by the rate expected under a chance model. 1.0 means observed and expected recurrence match under this null; above 1 means excess recurrence relative to this null.

The steps below

1. load the data
2. build each author's publication history
3. cross-check the counting against Lennart's Prolog output
4. compute the observed rates
5. build the chance model, two versions
6. results
7. simulation ranges and conditional confidence intervals
8. why Q3 is not in this notebook

The original simulation uses seed 42. The uncertainty calculation also uses NumPy and a separate bootstrap seed. Run all cells to save both the original comparisons and the added intervals.

## Step 1: load the data

- the semiclean CSV, in `../A_data_and_rules/data/` and not in git, one row per paper with date, journal, year, topic and the author list
- Lennart's journal-to-publisher mapping from `../A_data_and_rules/results/` in the repo, so journals roll up to their parent publisher and BMC counts as Springer Nature
- one cleanup, a few papers list the same author twice and we keep each author once per paper

In [1]:
# Fetch verified shared inputs; the existing analysis paths stay the same.
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parent
if not (project_root / "data-manifest.json").is_file():
    raise RuntimeError("Run this notebook with its working directory set to B_opportunities_and_analysis/.")
sys.path.insert(0, str(project_root))
from project_data import ensure_data
ensure_data("q1-q2", root=project_root)

import csv, json, random, sys
from bisect import bisect
from collections import defaultdict

csv.field_size_limit(sys.maxsize)   # the authorships json column is longer than the csv default allows

works = {}          # work_id -> (date, journal, year, topic)
work_authors = {}   # work_id -> [author_id, ...] without duplicates
with open("../A_data_and_rules/data/openalex_ai_semiclean_v1_0.csv", encoding="utf-8-sig", newline="") as f:
    for r in csv.DictReader(f):
        w = r["work_id"]
        works[w] = (r["publication_date"], r["journal_id"], int(r["publication_year"]), r["primary_topic_id"])
        seen = set(); auths = []
        for a in json.loads(r["authorships_json"]):
            if a.get("author_id") and a["author_id"] not in seen:   # some papers list an author twice, keep one
                seen.add(a["author_id"]); auths.append(a["author_id"])
        work_authors[w] = auths

parent = {}   # journal -> parent publisher, None when unresolved (1 journal)
with open("../A_data_and_rules/results/openalex_three_path_v1_0/journal_parent_publishers.csv", newline="", encoding="utf-8-sig") as f:
    for r in csv.DictReader(f):
        parent[r["journal_id"]] = None if r["is_unresolved"] == "True" else r["parent_publisher_id"]

print(f"{len(works):,} papers, {len(parent)} journals mapped to publishers")

verified A_data_and_rules/data/openalex_ai_semiclean_v1_0.csv
verified A_data_and_rules/results/openalex_three_path_v1_0/journal_parent_publishers.csv
verified A_data_and_rules/results/openalex_three_path_v1_0/pathway_flags.csv


27,400 papers, 64 journals mapped to publishers


## Step 2: build each author's publication history

- everything below asks some version of "had this author already done X before this paper", so the papers have to be in time order per author
- a paper with 3 authors lands in 3 histories, which is where the author-paper pairs come from, the same unit as Lennart's run
- only strictly earlier dates count as before. Two papers on the same date do not see each other, because OpenAlex fills missing day and month with January 1 and the order inside such a date is unknowable

In [2]:
papers_of = defaultdict(list)   # author -> [(date, work, journal)]
for w, (d, j, y, t) in works.items():
    for a in work_authors[w]:   # every author of a paper gets their own row
        papers_of[a].append((d, w, j))
for a in papers_of:
    papers_of[a].sort()   # dates are ISO strings, sorting puts earlier papers first

n = sum(len(v) for v in papers_of.values())
print(f"{len(papers_of):,} authors, {n:,} author-paper pairs")

82,539 authors, 110,654 author-paper pairs


## Step 3: cross-check against the Prolog output

Rebuild the three paper-level flags in Python and compare every author-paper row with the Prolog output.

- journal_path: the author published earlier in this journal
- publisher_path: the author published earlier with this parent publisher through at least one other journal
- coauthor_path: someone else on this paper published earlier in this journal

Agreement on all 110,654 rows verifies agreement between the two rule implementations. It does not establish complete publication histories or causal validity.


In [3]:
# walk each author's papers in date order and remember what was seen before
my = {}  # (work, author) -> [journal_path, publisher_path, coauthor_path]
for a, lst in papers_of.items():
    seen_j = set()               # journals seen at earlier dates
    seen_pj = defaultdict(set)   # parent publisher -> journals it was seen through
    i = 0
    while i < len(lst):
        d0 = lst[i][0]; grp = []
        while i < len(lst) and lst[i][0] == d0:   # papers sharing a date form one group
            grp.append(lst[i]); i += 1
        for d, w, j in grp:                        # flags first, against strictly earlier papers only
            p = parent.get(j)
            my[(w, a)] = [j in seen_j,
                          p is not None and p in seen_pj and any(x != j for x in seen_pj[p]),   # parent seen through another journal
                          False]
        for d, w, j in grp:                        # then the group becomes history
            seen_j.add(j)
            p = parent.get(j)
            if p is not None:
                seen_pj[p].add(j)

# coauthor flag in two passes
first_in_journal = {}   # (author, journal) -> earliest date they appeared there
for a, lst in papers_of.items():
    for d, w, j in lst:
        k = (a, j)
        if k not in first_in_journal or d < first_in_journal[k]:
            first_in_journal[k] = d
for w, (d, j, y, t) in works.items():
    # "9999" sorts after every real date, so missing means never
    early = [b for b in work_authors[w] if first_in_journal.get((b, j), "9999") < d]   # authors of this paper who were in the journal before
    for a in work_authors[w]:
        my[(w, a)][2] = any(b != a for b in early)   # at least one of them must be someone else

In [4]:
lf = {}   # the same flags as Prolog computed them
with open("../A_data_and_rules/results/openalex_three_path_v1_0/pathway_flags.csv", newline="", encoding="utf-8-sig") as f:
    for r in csv.DictReader(f):
        lf[(r["focal_work_id"], r["focal_author_id"])] = (
            r["journal_path"] == "True", r["publisher_path"] == "True", r["coauthor_path"] == "True")

assert set(my) == set(lf)   # both sides must cover exactly the same pairs
mismatches = [k for k in lf if tuple(my[k]) != lf[k]]
cnt = lambda d, i: sum(1 for v in d.values() if v[i])   # how many pairs have flag number i
print(f"journal {cnt(my,0):,} vs {cnt(lf,0):,} | publisher {cnt(my,1):,} vs {cnt(lf,1):,} | coauthor {cnt(my,2):,} vs {cnt(lf,2):,}")
print(f"mismatching rows: {len(mismatches)}")
assert not mismatches and cnt(my,0) == 12325 and cnt(my,1) == 4919 and cnt(my,2) == 26136   # pinned to his run, drift fails loudly
print("every row matches, the flags are confirmed")

journal 12,325 vs 12,325 | publisher 4,919 vs 4,919 | coauthor 26,136 vs 26,136
mismatching rows: 0
every row matches, the flags are confirmed


## Step 4: the observed rates

Three statistics, each a share of all author-paper pairs.

- Q1, the author was already in this journal. Anna publishes in Sensors and had a Sensors paper two years ago, that pair counts
- Q2 wide, the author was already with this parent publisher through any journal. This picks up essentially every Q1 case, since the same journal means the same publisher. Essentially, because 6 Q1 pairs sit on the one journal without a resolvable publisher and drop out here
- Q2 narrow, the author was already with this parent publisher through at least one other journal. This is the version that asks about the publisher rather than the journal, although it does not require the journal to be new, and 1,468 of the 4,919 narrow pairs are journal returns as well

All three are needed because Q2 wide on its own cannot separate publisher loyalty from journal loyalty.

In [5]:
# same walk as step 3, now counting the three statistics instead of storing flags
q1_pos = q2w = q2n = 0
eligible = 0        # pairs that had any earlier paper at all, only those can repeat
q1_eligible = 0     # of those, how many are returns
overlap = 0         # narrow pairs that are also Q1 returns
for a, lst in papers_of.items():
    seen_j = set(); seen_p = set(); seen_pj = defaultdict(set); i = 0
    history = 0     # how many papers of this author lie strictly earlier
    while i < len(lst):
        d0 = lst[i][0]; grp = []
        while i < len(lst) and lst[i][0] == d0:
            grp.append(lst[i]); i += 1
        for d, w, j in grp:
            if history: eligible += 1
            f1 = j in seen_j
            p = parent.get(j)
            f2 = p is not None and p in seen_pj and any(x != j for x in seen_pj[p])
            if f1: q1_pos += 1
            if f1 and history: q1_eligible += 1
            if p is not None and p in seen_p: q2w += 1   # parent seen before, same journal allowed
            if f2: q2n += 1   # parent seen through at least one other journal, this one may be known too
            if f1 and f2: overlap += 1
        for d, w, j in grp:
            history += 1
            seen_j.add(j); p = parent.get(j)
            if p is not None: seen_p.add(p); seen_pj[p].add(j)

obs = (q1_pos / n, q2w / n, q2n / n)
print(f"Q1 observed:        {obs[0]:.4f}  ({q1_pos:,} of {n:,} pairs)")
print(f"Q2 wide observed:   {obs[1]:.4f}  ({q2w:,} pairs)")
print(f"Q2 narrow observed: {obs[2]:.4f}  ({q2n:,} pairs)")
print(f"\nof the {q2n:,} narrow pairs, {overlap:,} are journal returns as well, {q2n - overlap:,} are not")
print(f"decomposition: {q1_pos:,} Q1 + {q2n - overlap:,} narrow-not-Q1 - 6 on the unresolved journal = {q2w:,} wide")
print(f"\n{eligible:,} of {n:,} pairs ({eligible/n:.0%}) had an earlier paper at all, the rest cannot repeat by construction")
print(f"among those eligible pairs Q1 is {q1_eligible/eligible:.0%}, not {obs[0]:.0%}")

Q1 observed:        0.1114  (12,325 of 110,654 pairs)
Q2 wide observed:   0.1425  (15,770 pairs)
Q2 narrow observed: 0.0445  (4,919 pairs)

of the 4,919 narrow pairs, 1,468 are journal returns as well, 3,451 are not
decomposition: 12,325 Q1 + 3,451 narrow-not-Q1 - 6 on the unresolved journal = 15,770 wide

26,790 of 110,654 pairs (24%) had an earlier paper at all, the rest cannot repeat by construction
among those eligible pairs Q1 is 46%, not 11%


Two notes before the chance model.

- the 11% is out of all pairs, and 81% of authors have exactly one paper in the corpus, so most pairs never had anything to repeat
- counting only the pairs that had an earlier paper, Q1 is 46%
- both numbers are right, they answer different questions, and the chance model carries the same structural zeros so the ratio is not affected either way
- is 11% high? I cannot tell from that number alone. If one journal published half the field then plenty of repeats would happen from volume, which is what the next step is for

## Step 5: the chance model

For every author, preserve the number of papers, dates, years and primary-topic categories. Redraw the journal of each author-paper position and rebuild that author's simulated history. Recount the same Q1, Q2 wide and Q2 narrow statistics, then average their rates across 100 simulations (seed 42).

Two versions are applied to all three statistics:

- A: journal probabilities proportional to the number of corpus papers in that journal and year.
- B: journal probabilities proportional to the number of corpus papers in that journal, year and primary-topic category.

Both Q1 and Q2 therefore account for topic category under B. Neither uses Pierre's continuous historical author-journal topic match T or his separate Q1 Intra statistic.

Draws are **with replacement**: the cumulative weights stay fixed after every draw, so the same journal can recur. This defines an independent weighted-assignment null; it does not exactly preserve journal totals in each simulated world. A permutation of a finite collection of paper/journal slots could preserve specified totals without preventing repeats, but would be a different null requiring its own construction and check. Removing each journal after its first draw would instead forbid the recurrence being studied.

Weights come from paper counts, while draws occur separately per author-paper pair. The same coauthored paper can receive different journals in different authors' simulated histories. Author productivity is preserved; the shared-paper structure of the full corpus is not. A single shared draw per paper would be another possible design, not an implemented sensitivity.

B accounts for coarse topic concentration, not necessarily all thematic selection. The topic category may itself carry venue information. Neither null identifies a true attachment parameter, and the ratios are not lower and upper bounds on one. These are the implemented comparisons; we have no evidence that all alternatives were explicitly considered and rejected when the original code was written.

[Inputs, outputs and reasoning](README.md)


In [6]:
def build_dist(keyfun):
    """journal size table per cell (cell = year, or year+topic)"""
    cells = defaultdict(lambda: defaultdict(int))
    for w, tup in works.items():
        cells[keyfun(tup)][tup[1]] += 1
    out = {}
    for k, cts in cells.items():
        js, wts = zip(*sorted(cts.items()))
        cum = []; s = 0
        for x in wts:
            s += x; cum.append(s)   # cumulative sums so bisect can draw a weighted random journal
        out[k] = (js, cum, s)
    return out

def null_rates(dist, keyfun, M=100, *, keep_draws=False):
    """redraw all journals M times, return average rates for Q1, Q2 wide, Q2 narrow"""
    rng = random.Random(42)   # fresh seed per version so each reproduces on its own
    acc = [0.0, 0.0, 0.0]
    draws = []
    for _ in range(M):
        rep = pw = pn = 0
        for a, lst in papers_of.items():
            seen_j = set(); seen_p = set(); seen_pj = defaultdict(set); i = 0
            while i < len(lst):
                d0 = lst[i][0]; grp = []
                while i < len(lst) and lst[i][0] == d0:
                    grp.append(lst[i]); i += 1
                drawn = []
                for d, w, j in grp:
                    js, cum, s = dist[keyfun(works[w])]
                    dj = js[bisect(cum, rng.random() * s)]   # size-weighted random journal
                    drawn.append(dj)
                    if dj in seen_j: rep += 1
                    p = parent.get(dj)
                    if p is not None:
                        if p in seen_p: pw += 1
                        if p in seen_pj and any(x != dj for x in seen_pj[p]): pn += 1
                for dj in drawn:                     # the date group becomes history afterwards
                    seen_j.add(dj); p = parent.get(dj)
                    if p is not None: seen_p.add(p); seen_pj[p].add(dj)
        draws.append((rep / n, pw / n, pn / n))
        acc[0] += rep / n; acc[1] += pw / n; acc[2] += pn / n   # running sums, divided by M below
    means = [x / M for x in acc]
    return (means, draws) if keep_draws else means

dist_year = build_dist(lambda t: t[2])                  # version A, cell = year
dist_topic = build_dist(lambda t: (t[2], t[3]))         # version B, cell = year and topic

# how restrictive is version B? cells with a single journal leave the paper no choice at all
degenerate = [k for k, v in dist_topic.items() if len(v[0]) == 1]
print(f"version B has {len(dist_topic)} year+topic cells, {len(degenerate)} of them contain a single journal "
      f"({sum(dist_topic[k][2] for k in degenerate)} papers, {sum(dist_topic[k][2] for k in degenerate)/len(works):.1%} of the corpus)")

e_year, draws_year = null_rates(dist_year, lambda t: t[2], keep_draws=True)
e_topic, draws_topic = null_rates(dist_topic, lambda t: (t[2], t[3]), keep_draws=True)
print(f"expected under A (year):        Q1 {e_year[0]:.4f}, Q2 wide {e_year[1]:.4f}, Q2 narrow {e_year[2]:.4f}")
print(f"expected under B (year+topic):  Q1 {e_topic[0]:.4f}, Q2 wide {e_topic[1]:.4f}, Q2 narrow {e_topic[2]:.4f}")

version B has 470 year+topic cells, 22 of them contain a single journal (25 papers, 0.1% of the corpus)


expected under A (year):        Q1 0.0361, Q2 wide 0.0706, Q2 narrow 0.0383
expected under B (year+topic):  Q1 0.0501, Q2 wide 0.0843, Q2 narrow 0.0400


## Step 6: results, observed divided by expected

In [7]:
print("                                   A: year null   B: year+topic null")
for i, nm in enumerate(["Q1 journal repeat               ",
                        "Q2 wide (same parent, any)      ",
                        "Q2 narrow (same parent, other)  "]):
    print(f"{nm}   {obs[i]/e_year[i]:.2f}           {obs[i]/e_topic[i]:.2f}")

# pinned so a broken rerun fails instead of printing wrong numbers
assert abs(obs[0]/e_year[0] - 3.09) < 0.05 and abs(obs[0]/e_topic[0] - 2.22) < 0.05   # Q1
assert abs(obs[1]/e_year[1] - 2.02) < 0.05 and abs(obs[1]/e_topic[1] - 1.69) < 0.05   # Q2 wide
assert abs(obs[2]/e_year[2] - 1.16) < 0.05 and abs(obs[2]/e_topic[2] - 1.11) < 0.05   # Q2 narrow
assert (q1_pos, q2w, q2n, overlap, eligible) == (12325, 15770, 4919, 1468, 26790)     # the counts behind the story

                                   A: year null   B: year+topic null
Q1 journal repeat                  3.09           2.22
Q2 wide (same parent, any)         2.02           1.69
Q2 narrow (same parent, other)     1.16           1.11


## What I take from the table

For Q1, I see more journal recurrence than the two comparison models predict: 3.09 against the year null and 2.22 against the year+topic null. That is the conclusion I can defend. I cannot turn it into a claim of topic-independent loyalty.

For Q2, I need to keep the definitions apart:

- The wide ratios, 2.02 and 1.69, include journal returns. I cannot use them to isolate recurrence at the publisher level beyond Q1.
- The narrow ratios, 1.16 and 1.11, still include 1,468 journal returns among their 4,919 cases. I have not calculated a separate matching null for the remaining 3,451 cases.
- Being close to one does not establish that a publisher association is absent. The conditional author-bootstrap intervals below quantify one part of their uncertainty.

What I still need to be clear about

- The spread across the 100 redraws is not a confidence interval for the observed/expected ratio.
- I use current publisher ownership for every year, so historical ownership changes can be misclassified.
- Our history begins in 2015 and covers only the selected journals. A first paper here need not be an author's first paper elsewhere.
- The null uses paper-volume weights and separate author-paper draws. It does not preserve all coauthorship or journal-total constraints.


## Step 7: two different kinds of interval

I first show the middle 95% of the **100 simulated return rates**. This describes variation under our null model; its tail estimates are rough with 100 runs. It is not a confidence interval for the observed/expected ratio.

For the ratio, I use **4,000 bootstrap samples of whole authors** (seed 20260910). Each sample draws the original number of authors with replacement. All papers of a selected author stay together. I recompute observed and expected counts together, then divide their totals. I do not average author-specific ratios. The common number of author-paper rows cancels in that division.

For this frozen corpus, the exact ratio is a fixed description. The bootstrap asks how it would vary across hypothetical samples of comparable author histories.

These are **conditional 95% percentile confidence intervals**: journal/year/topic weights, publisher mapping and the corpus definition stay fixed. Authors are treated as independent sampling units. This handles repeated observations of an author, but not dependence between different authors sharing papers, uncertainty in the estimated weights, or missing publication history. It does not establish causal validity or coverage for the whole research population. See [cluster bootstrap](https://www.stata.com/support/faqs/statistics/bootstrap-with-panel-data/).

To avoid adding simulation noise to the denominator, I calculate the same null's expectation directly. Given a current journal j, the probability of no earlier matching journal is the product of `(1 - earlier probability of j)`. For Q2 wide, replace j by its publisher. For Q2 narrow, use earlier probability of that publisher **minus** probability of j. Subtract each product from 1 and average over the current journal's probabilities. Same-date papers still do not count as earlier history.

This changes the numerical evaluation of the expectation, not the null model. The table keeps the original 100-run ratios beside the ratios with exact expectations, so any rounding change is visible. The confidence interval belongs to the latter.

In [8]:
import numpy as np

labels = ('Q1', 'Q2 wide', 'Q2 narrow')
print('Middle 95% of simulated return rates, not ratio confidence intervals')
for model_name, draws in [('year', draws_year), ('year+topic', draws_topic)]:
    bounds = np.quantile(draws, [0.025, 0.975], axis=0)
    for k, label in enumerate(labels):
        print(f'{model_name:10s} {label:10s}: observed {obs[k]:.2%}; '
              f'null range {bounds[0,k]:.2%} to {bounds[1,k]:.2%}')


def expected_by_author(histories, paper_data, publisher_of, dist, keyfun):
    """Expected Q1/wide/narrow counts under the existing independent-draw null."""
    journals = sorted({j for js, _, _ in dist.values() for j in js})
    index = {j: k for k, j in enumerate(journals)}
    publishers = [publisher_of.get(j) for j in journals]
    known_publisher = np.array([p is not None for p in publishers])
    same_publisher = np.array([[p is not None and p == q for q in publishers]
                               for p in publishers], dtype=float)
    probabilities = {}
    for key, (js, cum, total) in dist.items():
        p = np.zeros(len(journals))
        p[[index[j] for j in js]] = np.diff([0, *cum]) / total
        publisher_p = same_publisher @ p
        # Rows describe a prior match for each possible current journal.
        match_p = np.vstack((p, publisher_p, np.clip(publisher_p - p, 0, 1)))
        current_p = np.vstack((p, p * known_publisher, p * known_publisher))
        probabilities[key] = current_p, match_p
    result = np.zeros((len(histories), 3))
    for row, history in enumerate(histories.values()):
        if not history or history[0][0] == history[-1][0]:
            continue  # no strictly earlier date, including all single-paper authors
        no_prior_match = np.ones((3, len(journals)))
        i = 0
        while i < len(history):
            date = history[i][0]
            group = []
            while i < len(history) and history[i][0] == date:
                group.append(probabilities[keyfun(paper_data[history[i][1]])])
                i += 1
            for current_p, _ in group:
                result[row] += np.sum(current_p * (1 - no_prior_match), axis=1)
            for _, match_p in group:
                no_prior_match *= 1 - match_p
    return result


def observed_by_author(histories, publisher_of):
    result = np.zeros((len(histories), 3))
    for row, history in enumerate(histories.values()):
        seen_j = set(); seen_pj = defaultdict(set); i = 0
        while i < len(history):
            date = history[i][0]; group = []
            while i < len(history) and history[i][0] == date:
                group.append(history[i]); i += 1
            for _, _, j in group:
                p = publisher_of.get(j)
                result[row] += (j in seen_j,
                                p is not None and p in seen_pj,
                                p is not None and any(x != j for x in seen_pj.get(p, ())))
            for _, _, j in group:
                seen_j.add(j)
                p = publisher_of.get(j)
                if p is not None: seen_pj[p].add(j)
    return result


def bootstrap_ratios(observed_counts, expected_counts, reps=4000, seed=20260910):
    """Paired whole-author resampling; expected_counts has model x author x statistic."""
    observed_counts = np.asarray(observed_counts, dtype=float)
    expected_counts = np.asarray(expected_counts, dtype=float)
    n_authors = len(observed_counts)
    if expected_counts.ndim != 3 or expected_counts.shape[1:] != observed_counts.shape:
        raise ValueError('Observed and expected counts must cover the same authors/statistics.')
    # Zero-contribution authors can be represented as one multinomial category.
    # They still have their full sampling probability; no authors are excluded.
    active = (observed_counts != 0).any(axis=1) | (expected_counts != 0).any(axis=(0, 2))
    probs = np.append(np.full(active.sum(), 1 / n_authors), (~active).sum() / n_authors)
    obs_active = observed_counts[active]
    exp_active = expected_counts[:, active]
    rng = np.random.default_rng(seed)
    ratios = np.empty((reps, expected_counts.shape[0], observed_counts.shape[1]))
    for start in range(0, reps, 64):
        end = min(start + 64, reps)
        weights = rng.multinomial(n_authors, probs, size=end-start)[:, :-1]
        numerator = weights @ obs_active
        for model in range(expected_counts.shape[0]):
            denominator = weights @ exp_active[model]
            if np.any(denominator <= 0):
                raise ValueError('A bootstrap sample has no expected returns; interval undefined.')
            ratios[start:end, model] = numerator / denominator
    return ratios

observed_counts = observed_by_author(papers_of, parent)
assert np.array_equal(observed_counts.sum(axis=0), [q1_pos, q2w, q2n])
expected_counts = np.stack([
    expected_by_author(papers_of, works, parent, dist_year, lambda t: t[2]),
    expected_by_author(papers_of, works, parent, dist_topic, lambda t: (t[2], t[3])),
])
exact_rates = expected_counts.sum(axis=1) / n
exact_ratios = observed_counts.sum(axis=0) / expected_counts.sum(axis=1)
bootstrap_draws = bootstrap_ratios(observed_counts, expected_counts)
confidence_bounds = np.quantile(bootstrap_draws, [0.025, 0.975], axis=0)
print('\nRatio confidence intervals: conditional author bootstrap, 4,000 samples')
print('Model       Statistic   100-run ratio   Exact-null ratio   Conditional 95% CI')
for m, (model_name, simulated) in enumerate([('year', e_year), ('year+topic', e_topic)]):
    for k, label in enumerate(labels):
        lo, hi = confidence_bounds[:,m,k]
        print(f'{model_name:10s} {label:10s} {obs[k]/simulated[k]:13.4f} '
              f'{exact_ratios[m,k]:18.4f}   [{lo:.4f}, {hi:.4f}]')
print('Weights fixed. Whole authors resampled. Shared-paper dependence between authors not covered.')


Middle 95% of simulated return rates, not ratio confidence intervals
year       Q1        : observed 11.14%; null range 3.53% to 3.69%
year       Q2 wide   : observed 14.25%; null range 6.94% to 7.16%
year       Q2 narrow : observed 4.45%; null range 3.73% to 3.93%
year+topic Q1        : observed 11.14%; null range 4.91% to 5.10%
year+topic Q2 wide   : observed 14.25%; null range 8.32% to 8.54%
year+topic Q2 narrow : observed 4.45%; null range 3.88% to 4.12%



Ratio confidence intervals: conditional author bootstrap, 4,000 samples
Model       Statistic   100-run ratio   Exact-null ratio   Conditional 95% CI
year       Q1                3.0880             3.0936   [3.0343, 3.1541]
year       Q2 wide           2.0196             2.0200   [1.9923, 2.0476]
year       Q2 narrow         1.1603             1.1590   [1.1145, 1.2033]
year+topic Q1                2.2231             2.2229   [2.1815, 2.2647]
year+topic Q2 wide           1.6906             1.6905   [1.6682, 1.7130]
year+topic Q2 narrow         1.1118             1.1123   [1.0725, 1.1526]
Weights fixed. Whole authors resampled. Shared-paper dependence between authors not covered.


## Step 8: why Q3 needs a different table

Q3 compares journal entry with and without an earlier coauthor connection. Its denominator must include eligible journals an author did not enter. This notebook contains only observed publications.

The calculation below gives 0.68 among these publication rows. It asks about first-entry papers among publications with versus without a same-paper coauthor flag. It neither answers Q3 nor shows that coauthors prevent entry. The population, denominator and connection definition all differ from the annual opportunity analysis.

The annual earlier-collaborator seed, entry/non-ride outcomes and model comparisons are implemented in [q3_baselines.ipynb](q3_baselines.ipynb). I keep the calculation below to show why I need the opportunity denominator and the earlier-collaborator definition for Q3.


In [9]:
f1c1 = sum(1 for v in my.values() if not v[0] and v[2])   # first entries among pairs with the coauthor flag
f1c0 = sum(1 for v in my.values() if not v[0] and not v[2])   # first entries among pairs without it
c1 = cnt(my, 2); c0 = n - c1
print(f"P(first entry | coauthor flag) = {f1c1/c1:.3f}")
print(f"P(first entry | no flag)       = {f1c0/c0:.3f}")
print(f"ratio = {(f1c1/c1)/(f1c0/c0):.3f}  (not Q3, see text above)")

P(first entry | coauthor flag) = 0.656
P(first entry | no flag)       = 0.961
ratio = 0.683  (not Q3, see text above)


## Scope and possible extensions

The descriptive Q1/Q2 comparisons above are implemented. The annual opportunity set, earlier-collaborator seed and ride/non-ride split are implemented separately in Q3; they are not missing work for this notebook.

Stronger or different Q1/Q2 questions would need additional work:

- historical continuous T for return opportunities and alternative journals
- historical publisher ownership
- uncertainty beyond the conditional author bootstrap below, including estimated journal weights and dependence between coauthors
- a null for narrow-and-not-Q1 alone (3,451 observed cases)
- comparisons using author-paper volume weights, exact stratum totals, or one shared journal draw per coauthored paper

The current null already holds the number of papers per author fixed. Corpus papers without usable author IDs contribute to paper-volume weights but not author-paper rows. These choices and limits should be explained rather than described as having already been tested away.
